<a href="https://colab.research.google.com/github/IshaCodex/OIBSIP/blob/main/DataAnalytics-L1-CleaningData/Task3_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import the required libraries for data manipulation and mathematical operations.

In [ ]:
import pandas as pd
import numpy as np

Load the raw dataset into a Pandas DataFrame.

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/OASIS_DataAnalytics/dirty_cafe_sales.csv')

# Data Quality Report

Check the data types of all columns. Currently, all columns will likely show as "object" (text) because of the embedded error strings.

In [4]:
print(df.dtypes)

Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object


Count the initial null (missing) values across all columns.

In [5]:
print(df.isnull().sum())

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


Check for duplicate rows in the dataset.

In [6]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


Check the value ranges and descriptive statistics. (Note: Because numeric columns are currently formatted as objects, this will only show counts and frequencies until we fix the data types).

In [7]:
print(df.describe(include='all'))

       Transaction ID   Item Quantity Price Per Unit Total Spent  \
count           10000   9667     9862           9821        9827   
unique          10000     10        7              8          19   
top       TXN_9226047  Juice        5            3.0         6.0   
freq                1   1171     2013           2429         979   

        Payment Method  Location Transaction Date  
count             7421      6735             9841  
unique               5         4              367  
top     Digital Wallet  Takeaway          UNKNOWN  
freq              2291      3022              159  


# Data Type Correction

The numeric columns contain string text like "ERROR" and "UNKNOWN". I am using pd.to_numeric with errors='coerce'. This forces the column into a float data type and automatically replaces any unreadable text strings with NaN (null) values.

In [11]:
numeric_columns = ['Quantity', 'Price Per Unit', 'Total Spent']

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

Convert the Transaction Date column to a datetime format. The errors='coerce' parameter handles any invalid date strings by turning them into NaT (Not a Time).

In [12]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# Standardisation

The categorical columns contain strings like "ERROR" and "UNKNOWN" that pandas didn't catch during numeric conversion. I will standardize these by replacing them with true NaN values so they can be imputed properly in the next step.

In [13]:
categorical_columns = ['Item', 'Payment Method', 'Location']

for col in categorical_columns:
    df[col] = df[col].replace(['UNKNOWN', 'ERROR'], np.nan)

# Missing Data Handling

For the numeric columns (Quantity, Price Per Unit, Total Spent), I am choosing median imputation. The median is highly robust against extreme outliers, making it a safer choice than the mean for transactional data where a single massive purchase could skew the average.

In [14]:
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

For categorical text data and transaction dates, calculating a median is impossible. I am choosing mode imputation (filling with the most frequent value) to handle missing items, locations, payment methods, and dates.

In [15]:
columns_for_mode = categorical_columns + ['Transaction Date']

for col in columns_for_mode:
    df[col] = df[col].fillna(df[col].mode()[0])

# Duplicate Removal

Drop any duplicate rows from the dataset. Even if the initial count was zero, applying this function ensures that any hidden duplicates exposed during the standardization phase are eliminated.

In [16]:
df = df.drop_duplicates()
print("Duplicates remaining:", df.duplicated().sum())

Duplicates remaining: 0


# Outlier Detection

I am using the Interquartile Range (IQR) method to detect anomalies in the Total Spent column.


In [17]:
Q1 = df['Total Spent'].quantile(0.25)
Q3 = df['Total Spent'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Total Spent'] < lower_bound) | (df['Total Spent'] > upper_bound)]
print(f"Number of outliers detected (and retained): {len(outliers)}")

Number of outliers detected (and retained): 259


# Before vs. After Summary Table

Generate a final summary table to confirm that all data types are now correct (floats and datetimes) and that zero null values remain in the dataset.

In [18]:
after_summary = pd.DataFrame({
    'Final Data Type': df.dtypes,
    'Final Null Count': df.isnull().sum()
})
print(after_summary)

                 Final Data Type  Final Null Count
Transaction ID            object                 0
Item                      object                 0
Quantity                 float64                 0
Price Per Unit           float64                 0
Total Spent              float64                 0
Payment Method            object                 0
Location                  object                 0
Transaction Date  datetime64[ns]                 0


Save the transformed, clean DataFrame to a new CSV file, ensuring the index column is not exported.

In [20]:
df.to_csv('/content/drive/MyDrive/Colab Notebooks/OASIS_DataAnalytics/cleaned_cafe_sales.csv', index=False)